#### Initialize

In [1]:
import sys
from pathlib import Path
import pandas as pd

HERE = Path.cwd()
PARENT = HERE.parent.parent.parent  # server/scripts
if str(PARENT) not in sys.path:
    sys.path.insert(0, str(PARENT))
LEVEL4_BUCKET_PATH = PARENT / "server/out/places_level4"
RANKED_BUCKET_PATH = PARENT / "server/out/places_ranked"
RANKED_BUCKET_PATH.mkdir(parents=True, exist_ok=True)
LEVEL4_BUCKET = [f for f in LEVEL4_BUCKET_PATH.rglob("*.csv") if f.is_file()]
DF_LEVEL4 = pd.concat([pd.read_csv(f) for f in LEVEL4_BUCKET], ignore_index=True)

#### Wilson Score


Two complementary methods are implemented side-by-side:

| Method | Formula | Strengths | Weaknesses |
|---|---|---|---|
| **Wilson Score** (primary) | Lower bound of 95 % CI for $\hat{p} = (\text{rating}-1)/4$ | Statistically robust; penalises sparse ratings correctly; used by Reddit & IMDB | Slightly complex; favours established places at high confidence |
| **Bayesian Average** (reference) | $\frac{v}{v+m} \cdot R + \frac{m}{v+m} \cdot C$ | Simple; naturally regresses to global mean | Less principled; sensitive to choice of $m$ |

**Wilson Score intuition:** a place with a 5.0 average from 3 reviews is *less trustworthy* than one with 4.8 from 400 reviews. The lower bound of the confidence interval captures exactly that — the more reviews, the tighter the interval and the higher the lower bound.

The `confidence` parameter controls how conservative the ranking is:
- **0.99** (default) — favours well-established places with many reviews  
- **0.95** — gives newer high-rated places a larger boost

In [2]:
from server.scripts.rank_places_wilson_score.wilson_score import wilson_score
MODE = "Moderate"  # "FavourRating", "Moderate", "FavourVolume"
CONFIDENCE = { # ← adjust: 0.99 = conservative, 0.95 = give newcomers more credit
    "FavourRating": 0.90,
    "Moderate": 0.95,
    "FavourVolume": 0.99
}
df_wilson = DF_LEVEL4.copy()
# df_wilson["capped_ratings"] = df_wilson["userRatingCount"].apply(lambda c: min(c, 1000))  # cap ratings to prevent outliers dominating

for idx, confidence_score in enumerate(CONFIDENCE.values()):
    df_wilson[f"wilson_{idx}"] = df_wilson.apply(
        lambda r: wilson_score(
            r["rating"], 
            r["userRatingCount"], 
            confidence_score
        ), axis=1
    )
    df_wilson[f"normal_{idx}"] = df_wilson[f"wilson_{idx}"].rank(pct=True)

df_wilson.sort_values("wilson_1", ascending=False, inplace=True)
df_wilson.reset_index(drop=True, inplace=True)
df_wilson.index += 1  # 1-based rank index

print(f"Total ranked: {len(df_wilson)}  |  confidence: {CONFIDENCE[MODE]}")
display(df_wilson[(df_wilson["normal_1"] >= 0.9) 
    ][["displayName", "primaryTypeDisplayName", "wilson_1", "normal_1"]]
    .head(5)
)

Total ranked: 13092  |  confidence: 0.95


,displayName,primaryTypeDisplayName,wilson_1,normal_1
1,Tofu Vegan Charlotte Street,Restaurant,0.999375,1.000000
2,Ethical Bean Company Coffee Shop,Vegan Restaurant,0.999154,0.999924
3,Falafel Zaki Zaki,Vegan Restaurant,0.997816,0.999847
4,Eye Falafel,Falafel Restaurant,0.997740,0.999771
5,Brazilicious Churros,Restaurant,0.996218,0.999694


#### Export

In [3]:
df_wilson.to_csv(RANKED_BUCKET_PATH / f"places_scored_level_1.csv", index=False)